In [ ]:
# Empty cell required for the "ruff" extension

# Feature model analysis

Feature models are techniques to extract information from data (in this case, a string).
In the context of near-duplicates detection feature models are used by the Simhash algorithm to create fixed-length fingerprints.
Therefore, it's crucial to find the optimal feature model (and its parameters).
In this project, 4 feature models are considered: bag_of_words, shingles, tf-idf and embeddings.

## bag_of_words

The simplest feature model of the four. For a given string, its features are the words it's made of.
Example: "This is a string" -> ("This", "is", "a", "string")
Hence, two strings are "near-equal" iff they contain about the same words, in any order.

It's simple and quick, but cannot detect locality.
Example: "The bank is after the office" is exactly equal to "The office is after the bank"

## shingles

Strings are sliced using a sliding window algorithm, with a certain _window_size_.
Example: "This is a string" -> ("Thi", "his", " is", "is ", ...) with _window_size_=3
As before, two strings are "near-equal" iff they contain about the same shingles, in any order.

Compared to _bag_of_words_, it generates many more elements but it can detect locality (to some degree).
As a middle-ground, the "word-based" shingle feature model is also considered here: it iterates over words, instead of characters.
Example: "This is a string" -> ("This is", "is a", "a string") with _window_size_=2

## tf-idf

A more refined version of _bag_of_words_, which values "rare" words more than "common" ones.
Each word in each string has a weight (0,1).

## embeddings

Generated using the transformer-based model Qwen3-embedding:0.6b.
Two strings are "near-equal" semantically if their embedding is similar.
Despite its great performance, it tends to be computationally more expensive than other models.


In [ ]:
import sys
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.datasets import read_paws, read_wikipedia
from src.near_duplicates import near_duplicates_benchmark
from src.fingerprint import FingerprintParams, create_fingerprints
from src.fingerprint import FINGERPRINT_SIZES

spark = get_spark()

%load_ext autoreload
%autoreload 2

In [ ]:
from numpy import var


DATASETS = ("wikipedia", "paws")
FEATURE_MODELS = ["tf-idf", "shingles", "bag_of_words"]
F_SIZES = [32, 64, 128, 256, 512]
SHINGLE_CONFIGS = [
    FingerprintParams(shingle_size=3, over_words=False),
    FingerprintParams(shingle_size=5, over_words=False),
    FingerprintParams(shingle_size=7, over_words=False),
    FingerprintParams(shingle_size=3, over_words=True),
    FingerprintParams(shingle_size=5, over_words=True),
    FingerprintParams(shingle_size=7, over_words=True)
]

### CONFIGURABLE PARAMS ###
DATASET_NAME = "wikipedia"
COMPUTE = False
K_SAMPLE = 3
SAMPLE_RATIO = 0.1
### END CONFIGURABLE PARAMS ###

assert([fs in FINGERPRINT_SIZES for fs in F_SIZES])
assert(DATASET_NAME in DATASETS)
assert(K_SAMPLE > 0 and K_SAMPLE < min(F_SIZES))

COL_IN1 = "data1"
COL_IN2 = "data2"
COL_OUT1 = "fingerprint1"
COL_OUT2 = "fingerprint2"
TRUTH_COL = "is_near_duplicate"

dataset = None
match DATASET_NAME:
    case "paws":
        dataset = read_paws()
    case "wikipedia":
        dataset = read_wikipedia(variant="10percent")

dataset = dataset.sample(SAMPLE_RATIO)
dataset.persist()
dataset_sample_size = dataset.count()
sample_near_dup_count = dataset.filter("is_near_duplicate == True").count()
print(f'Selected dataset: {DATASET_NAME}, with sample ratio of {SAMPLE_RATIO}.')
print(f"This sample has {sample_near_dup_count} near-duplicate rows and {dataset_sample_size - sample_near_dup_count} NON near-duplicate rows.")

def feature_model_benchmark():
    # Iterate over model_feature(shingle_size) and fingeprint_size
    # For each, iterate over K, marking the DataFrame as persistent
    models = FEATURE_MODELS
    fingerprint_sizes = F_SIZES
    experiments = []
    results = []
    Ks = [i for i in range(0, fingerprint_sizes[-1], K_SAMPLE)]

    for m, fs in itertools.product(models, fingerprint_sizes):
        if m == "shingles":
            for conf in SHINGLE_CONFIGS:
                experiments.append({
                    "model": m,
                    "fingerprint_size": fs,
                    "other": conf
                })
        else:
            experiments.append({
                "model": m,
                "fingerprint_size": fs,
                "other": None
            })

    for exp in tqdm(experiments, desc="Feature model benchmark running..."):
        try:
            df_exp = dataset
            df_exp = create_fingerprints(
                df=df_exp,
                model=exp["model"],
                input_col=COL_IN1,
                output_col=COL_OUT1,
                fingerprint_size=exp["fingerprint_size"],
                other=exp["other"]
            )
            df_exp = create_fingerprints(
                df=df_exp,
                model=exp["model"],
                input_col=COL_IN2,
                output_col=COL_OUT2,
                fingerprint_size=exp["fingerprint_size"],
                other=exp["other"]
            )
            df_exp.persist()
            for K in Ks:
                if K <= int(exp["fingerprint_size"]/2):
                    metrics = near_duplicates_benchmark(
                        df=df_exp,
                        f1_col=COL_OUT1,
                        f2_col=COL_OUT2,
                        truth_col=TRUTH_COL,
                        K=K
                    )
                    row = {**exp, "K": K, **metrics.__dict__}
                    results.append(row)
            df_exp.unpersist()
            
        except Exception as e:
            print(exp, e)
            return None

    return pd.DataFrame(results)
# dataset.show(truncate=False)

In [ ]:
benchmark_df = pd.DataFrame()
filename = f"results/{DATASET_NAME}/feature_benchmark_{DATASET_NAME}.csv"
csv_file = Path(filename)
if csv_file.is_file() and not COMPUTE:
    benchmark_df = pd.read_csv(filename)
else:
    benchmark_df = feature_model_benchmark()
    benchmark_df.to_csv(filename)
benchmark_df

In [ ]:
# Plots: fingerprint_size to f1_score
# Best f1-scores by model
# Show best f1 score by (model, fingerprint_size)
# Show which K/fingerprint_size ratio has best f1 score by model
# Show top f1 score by (shingle_size, over_words)
assert benchmark_df is not None
top_fs_f1 = benchmark_df.sort_values(by=["fingerprint_size"])[["fingerprint_size", "f1_score"]].groupby(by=["fingerprint_size"]).agg({"f1_score": "max"})
top_fs_f1["f1_score"] = top_fs_f1["f1_score"].round(2)
plt.figure(figsize=(10,6))
plot = sns.barplot(
    data=top_fs_f1,
    x="fingerprint_size",
    y="f1_score",
    hue="f1_score",
    palette="deep"
)
plt.ylim(top_fs_f1["f1_score"].min() - 0.2, 1)
plt.title("Max f1_score for each fingerprint size")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
assert benchmark_df is not None
top_model = benchmark_df.sort_values(by=["f1_score"])[["model", "f1_score"]].groupby(by=["model"]).agg({"f1_score": "max"})
top_model["f1_score"] = top_model["f1_score"].round(2)
plt.figure(figsize=(10,6))
plot = sns.barplot(
    data=top_model,
    x="model",
    y="f1_score",
    hue="f1_score",
    palette="deep"
)
plt.ylim(top_model["f1_score"].min() - 0.2, 1)
plt.title("Best f1_score for each model")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
assert benchmark_df is not None

benchmark_df["K_ratio"] = benchmark_df["K"] / benchmark_df["fingerprint_size"]
top_K = (
    benchmark_df.sort_values(
        by=["model", "f1_score", "K_ratio"], ascending=[True, False, True]
    )
    .groupby("model", as_index=False)
    .first()
)

top_K["K_ratio"] = top_K["K_ratio"].round(2)
top_K["label"] = top_K.apply(
    lambda r: f"({int(r['K'])} on {int(r['fingerprint_size'])})", axis=1
)

plt.figure(figsize=(10, 6))
plot = sns.barplot(
    data=top_K,
    x="model",
    y="K_ratio",
    hue="model",
    palette="deep",
    legend=False,
)

for container, label in zip(plot.containers, top_K["label"]):
    plot.bar_label(container, labels=[label], padding=3) # type: ignore

plt.ylim(0, max(top_K["K_ratio"].max() * 1.15, 1.0))
plt.title("K to fingerprint size ratio of best performing configuration")
plt.xlabel("Feature model")
plt.ylabel("K to fingerprint size ratio")
plt.tight_layout()
plt.show()

In [ ]:
assert benchmark_df is not None
plot_df = benchmark_df.pivot_table(index="model", columns="fingerprint_size", values="f1_score", aggfunc="max").round(2)
plt.figure(figsize=(10,6))
plot = sns.heatmap(
    data=plot_df,
    annot=True,
    cmap="coolwarm",
    cbar=True
)
plt.title("F1_score by feature model / fingerprint size")
plt.xlabel("Fingerprint size")
plt.ylabel("Feature model")
plt.show()

In [ ]:
assert benchmark_df is not None
plot_df = pd.DataFrame()
plot_df[["shingle_size", "over_words"]] = benchmark_df["other"].dropna().str.extract(r"shingle_size=(\d+).*?over_words=(True|False)")
plot_df["f1_score"] = benchmark_df[["other", "f1_score"]].dropna()["f1_score"]
plot_df["shingle_size"] = plot_df["shingle_size"].astype(int)
plot_df["over_words"] = plot_df["over_words"].map({"True": "words", "False": "chars"})
plot_df = plot_df.pivot_table(index="shingle_size", columns="over_words", values="f1_score", aggfunc="max").round(2)
plt.figure(figsize=(10,6))
plot = sns.heatmap(
    data=plot_df,
    annot=True,
    cmap="coolwarm",
    cbar=True
)
plt.title("F1_score by shingle size / shingle method")
plt.xlabel("Shingle method")
plt.ylabel("Shingle size")
plt.show()

In [ ]:
assert dataset is not None
dataset.unpersist()
spark.stop()